
# 24 — Event Correlation Timeline

This notebook cross-references experiment outputs with external news/event signals to support
forensic interpretation.

It is designed to remain useful in two modes:

1. **API mode** — queries `NewsAPI` and `NewsData.io` using repository secrets  
2. **Offline mode** — if API keys are absent or requests fail, it still builds a clean event timeline scaffold from experiment outputs alone

## Inputs
- `outputs/gradient/coastal_gradient_daily_summary.csv`
- `outputs/clear_sky_snapshot/clear_sky_shortlist.csv`
- `outputs/wind_calm_test/wind_calm_candidate_days.csv`
- `outputs/back_trajectory/NHV_back_trajectory.csv` (optional)

## Outputs
- `outputs/event_correlation/event_timeline.csv`
- `outputs/event_correlation/event_articles.csv`
- `outputs/event_correlation/event_summary_by_day.csv`
- `outputs/event_correlation/event_correlation_plot.png`
- `outputs/event_correlation/event_correlation_manifest.json`


In [ ]:

DATE_FROM = "2024-12-26"
DATE_TO   = "2024-12-31"

OUTPUT_DIR = "outputs/event_correlation"
GRADIENT_CSV = "outputs/gradient/coastal_gradient_daily_summary.csv"
CLEAR_SKY_CSV = "outputs/clear_sky_snapshot/clear_sky_shortlist.csv"
WIND_CALM_CSV = "outputs/wind_calm_test/wind_calm_candidate_days.csv"
BACK_TRAJ_CSV = "outputs/back_trajectory/NHV_back_trajectory.csv"

# Places and keywords for event search
PLACE_TERMS = [
    "Newhaven",
    "Lewes",
    "Eastbourne",
    "Seaford",
    "Peacehaven",
    "Shoreham",
]
EVENT_TERMS = [
    "fire",
    "smoke",
    "industrial",
    "plant",
    "incinerator",
    "waste",
    "pollution",
    "air quality",
    "road closure",
    "traffic",
    "port",
    "gas leak",
]
MAX_ARTICLES_PER_SOURCE = 30


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import os, json, hashlib, time, re
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_read_csv(path_str):
    p = Path(path_str)
    if p.exists():
        return pd.read_csv(p), {"path": str(p), "sha256": sha256_file(p)}
    return pd.DataFrame(), None

print("UTC now:", datetime.now(timezone.utc).isoformat())
print("OUTPUT_DIR:", Path(OUTPUT_DIR).resolve())


In [ ]:

gradient, g_meta = safe_read_csv(GRADIENT_CSV)
clear_sky, c_meta = safe_read_csv(CLEAR_SKY_CSV)
wind_calm, w_meta = safe_read_csv(WIND_CALM_CSV)
back_traj, b_meta = safe_read_csv(BACK_TRAJ_CSV)

input_files = [x for x in [g_meta, c_meta, w_meta, b_meta] if x is not None]

print("gradient rows:", len(gradient))
print("clear_sky rows:", len(clear_sky))
print("wind_calm rows:", len(wind_calm))
print("back_traj rows:", len(back_traj))


In [ ]:

# Build experiment timeline scaffold
timeline_rows = []

if not gradient.empty and "date" in gradient.columns:
    for _, r in gradient.iterrows():
        timeline_rows.append({
            "date": str(r.get("date")),
            "source": "gradient",
            "site_id": r.get("site_id"),
            "site_name": r.get("site_name"),
            "signal_type": "coastal_gradient_day",
            "signal_strength": r.get("scenes", np.nan),
            "context": f"cloud={r.get('mean_cloud_cover_pct', np.nan)} wind={r.get('mean_wind_speed_ms', np.nan)}",
        })

if not clear_sky.empty and "date" in clear_sky.columns:
    for _, r in clear_sky.iterrows():
        timeline_rows.append({
            "date": str(r.get("date")),
            "source": "clear_sky",
            "site_id": r.get("site_id"),
            "site_name": r.get("site_name"),
            "signal_type": "clear_sky_candidate",
            "signal_strength": r.get("clarity_score", np.nan),
            "context": f"scene_count={r.get('scene_count', np.nan)}",
        })

if not wind_calm.empty and "date" in wind_calm.columns:
    for _, r in wind_calm.iterrows():
        timeline_rows.append({
            "date": str(r.get("date")),
            "source": "wind_calm",
            "site_id": r.get("site_id"),
            "site_name": r.get("site_name"),
            "signal_type": "calm_day_candidate",
            "signal_strength": r.get("mean_wind_speed_ms", np.nan),
            "context": f"cloud={r.get('mean_cloud_cover_pct', np.nan)}",
        })

timeline = pd.DataFrame(timeline_rows)
if not timeline.empty:
    timeline["date"] = pd.to_datetime(timeline["date"], errors="coerce").dt.date.astype("string")

print("Timeline rows:", len(timeline))
display(timeline.head(20))


In [ ]:

# API keys
NEWSAPI_KEY = os.getenv("NEWSAPI_KEY")
NEWSDATA_KEY = os.getenv("NEWSDATA_KEY") or os.getenv("NEWSDATA_IO_KEY")

QUERY_STRING = " OR ".join(PLACE_TERMS) + " AND (" + " OR ".join(EVENT_TERMS) + ")"
print("Query:", QUERY_STRING)
print("NewsAPI available:", bool(NEWSAPI_KEY))
print("NewsData available:", bool(NEWSDATA_KEY))


In [ ]:

def request_json_with_retry(url, params=None, headers=None, retries=3, timeout=45):
    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Request failed after {retries} attempts: {last_err}")

def normalize_newsapi(data):
    rows = []
    for a in data.get("articles", []):
        rows.append({
            "source_api": "newsapi",
            "published_utc": a.get("publishedAt"),
            "title": a.get("title"),
            "description": a.get("description"),
            "source_name": (a.get("source") or {}).get("name"),
            "url": a.get("url"),
            "content": a.get("content"),
        })
    return rows

def normalize_newsdata(data):
    rows = []
    for a in data.get("results", []):
        rows.append({
            "source_api": "newsdata",
            "published_utc": a.get("pubDate"),
            "title": a.get("title"),
            "description": a.get("description") or a.get("content"),
            "source_name": a.get("source_id") or a.get("source_name"),
            "url": a.get("link"),
            "content": a.get("content"),
        })
    return rows


In [ ]:

article_rows = []
api_log = []

# NewsAPI
if NEWSAPI_KEY:
    try:
        data = request_json_with_retry(
            "https://newsapi.org/v2/everything",
            params={
                "q": QUERY_STRING,
                "from": DATE_FROM,
                "to": DATE_TO,
                "language": "en",
                "sortBy": "publishedAt",
                "pageSize": MAX_ARTICLES_PER_SOURCE,
                "apiKey": NEWSAPI_KEY,
            },
        )
        article_rows.extend(normalize_newsapi(data))
        api_log.append({"api": "newsapi", "status": "ok", "rows": len(data.get("articles", []))})
    except Exception as e:
        api_log.append({"api": "newsapi", "status": "error", "error": str(e)})
else:
    api_log.append({"api": "newsapi", "status": "missing_key"})

# NewsData.io
if NEWSDATA_KEY:
    try:
        data = request_json_with_retry(
            "https://newsdata.io/api/1/news",
            params={
                "apikey": NEWSDATA_KEY,
                "q": QUERY_STRING,
                "country": "gb",
                "language": "en",
            },
        )
        rows = normalize_newsdata(data)[:MAX_ARTICLES_PER_SOURCE]
        article_rows.extend(rows)
        api_log.append({"api": "newsdata", "status": "ok", "rows": len(rows)})
    except Exception as e:
        api_log.append({"api": "newsdata", "status": "error", "error": str(e)})
else:
    api_log.append({"api": "newsdata", "status": "missing_key"})

articles = pd.DataFrame(article_rows)
if not articles.empty:
    articles["published_utc"] = pd.to_datetime(articles["published_utc"], utc=True, errors="coerce")
    articles["date"] = articles["published_utc"].dt.date.astype("string")
    articles["text_blob"] = (
        articles["title"].fillna("") + " " +
        articles["description"].fillna("") + " " +
        articles["content"].fillna("")
    )
else:
    articles = pd.DataFrame(columns=["source_api","published_utc","title","description","source_name","url","content","date","text_blob"])

print("Article rows:", len(articles))
display(articles.head(20))
display(pd.DataFrame(api_log))


In [ ]:

# Lightweight event scoring against place/event terms
def score_article(text):
    t = (text or "").lower()
    place_hits = [p for p in PLACE_TERMS if p.lower() in t]
    event_hits = [e for e in EVENT_TERMS if e.lower() in t]
    return len(place_hits) * 2 + len(event_hits), place_hits, event_hits

if not articles.empty:
    scored = []
    for _, r in articles.iterrows():
        score, place_hits, event_hits = score_article(r.get("text_blob"))
        scored.append({
            **r.to_dict(),
            "event_score": score,
            "place_hits": ",".join(place_hits),
            "event_hits": ",".join(event_hits),
        })
    articles = pd.DataFrame(scored).sort_values(["date","event_score"], ascending=[True, False]).reset_index(drop=True)

# Build day-level summary
if not articles.empty:
    article_day = (
        articles.groupby("date", dropna=False)
        .agg(
            article_count=("url","count"),
            max_event_score=("event_score","max"),
            top_title=("title","first"),
        )
        .reset_index()
    )
else:
    article_day = pd.DataFrame(columns=["date","article_count","max_event_score","top_title"])

summary = timeline.merge(article_day, on="date", how="left") if not timeline.empty else article_day.copy()
summary["article_count"] = summary.get("article_count", pd.Series(dtype=float)).fillna(0)
summary["max_event_score"] = summary.get("max_event_score", pd.Series(dtype=float)).fillna(0)

# Simple interpretation score
if not summary.empty:
    summary["forensic_interest_score"] = (
        pd.to_numeric(summary["signal_strength"], errors="coerce").fillna(0) +
        summary["article_count"].fillna(0) +
        summary["max_event_score"].fillna(0)
    )

print("Summary rows:", len(summary))
display(summary.head(30))


In [ ]:

# Save outputs
articles_csv = Path(OUTPUT_DIR) / "event_articles.csv"
timeline_csv = Path(OUTPUT_DIR) / "event_timeline.csv"
summary_csv = Path(OUTPUT_DIR) / "event_summary_by_day.csv"

articles.to_csv(articles_csv, index=False)
timeline.to_csv(timeline_csv, index=False)
summary.to_csv(summary_csv, index=False)

print("Saved:", articles_csv)
print("Saved:", timeline_csv)
print("Saved:", summary_csv)


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))

plotted = False
if not summary.empty and "date" in summary.columns:
    s = summary.groupby("date", dropna=False).agg(
        article_count=("article_count", "max"),
        max_event_score=("max_event_score", "max"),
        forensic_interest_score=("forensic_interest_score", "max"),
    ).reset_index()

    x = pd.to_datetime(s["date"])
    ax.plot(x, s["article_count"], marker="o", label="article_count")
    ax.plot(x, s["max_event_score"], marker="o", label="max_event_score")
    ax.plot(x, s["forensic_interest_score"], marker="o", label="forensic_interest_score")
    plotted = True

ax.set_title("Event correlation timeline")
ax.set_xlabel("Date")
ax.set_ylabel("Score")
if plotted:
    ax.legend()
fig.autofmt_xdate()
fig.tight_layout()

plot_path = Path(OUTPUT_DIR) / "event_correlation_plot.png"
fig.savefig(plot_path, dpi=150)
plt.show()

print("Saved:", plot_path)


In [ ]:

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": "event_correlation",
    "date_from": DATE_FROM,
    "date_to": DATE_TO,
    "query_string": QUERY_STRING,
    "input_files": input_files,
    "api_log": api_log,
    "timeline_rows": int(len(timeline)),
    "articles_rows": int(len(articles)),
    "summary_rows": int(len(summary)),
    "outputs": {
        "articles_csv": str(articles_csv),
        "timeline_csv": str(timeline_csv),
        "summary_csv": str(summary_csv),
        "plot_png": str(plot_path),
    },
    "notes": [
        "This notebook is a forensic event-correlation layer, not proof of causation by itself.",
        "It is intended to cross-reference anomaly days with plausible external events and incidents."
    ],
}
manifest_path = Path(OUTPUT_DIR) / "event_correlation_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved:", manifest_path)
